## Cell 1: Setup
**What this demonstrates**: Environment initialisation for Multi-Query RAG — Anthropic for sub-query decomposition and answer synthesis, OpenAI for embeddings, Chroma for retrieval.
**Expected output**: Setup confirmation with model names and configuration.

In [ ]:
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv

import os
import time
import pathlib

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# Haiku decomposes queries cheaply; Sonnet synthesises the final answer
DECOMPOSE_MODEL = 'claude-haiku-4-5-20251001'
ANSWER_MODEL    = 'claude-sonnet-4-6'
EMBED_MODEL     = 'text-embedding-3-small'
N_SUB_QUERIES   = 3   # sub-queries to generate per complex question
K_PER_QUERY     = 3   # chunks retrieved per sub-query
MAX_CONTEXT     = 8   # cap on unique chunks passed to generator

client     = Anthropic()
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Decompose model : {DECOMPOSE_MODEL}')
print(f'  Answer model    : {ANSWER_MODEL}')
print(f'  Sub-queries     : {N_SUB_QUERIES}')
print(f'  Chunks/query    : {K_PER_QUERY}')
print(f'  Max context     : {MAX_CONTEXT} unique chunks')

## Cell 2: Data — Basel III + Fintech Policy Corpus
**What this demonstrates**: Loading two complementary regulatory documents — Basel III for capital adequacy rules and fintech policy for lending criteria. This two-document corpus is ideal for Multi-Query RAG because a single complex query spans both documents, and sub-question decomposition retrieves targeted sections from each.
**Expected output**: Chunk counts per document and a preview of the cross-document query problem Multi-Query solves.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

# Basel III: capital ratios, buffer requirements, risk-weighted assets
# Fintech policy: lending criteria, LTV limits, borrower eligibility rules
# A query spanning both sits between two embedding clusters — decomposition fixes this
doc_files = {
    'basel_iii'     : 'basel_iii_excerpt.txt',
    'fintech_policy': 'fintech_policy.txt',
}

raw_docs: list[Document] = []
for source_name, filename in doc_files.items():
    text = (BASE_DIR / filename).read_text(encoding='utf-8')
    raw_docs.append(Document(page_content=text, metadata={'source': source_name}))
    print(f'  {source_name:<18}: {len(text):,} chars')

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks: list[Document] = splitter.split_documents(raw_docs)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='multi_query_corpus',
)

# Show chunk distribution across sources
from collections import Counter
source_counts = Counter(c.metadata['source'] for c in chunks)
print(f'\nTotal chunks: {len(chunks)}')
for src, cnt in source_counts.items():
    print(f'  {src:<18}: {cnt} chunks')
print()
print('Why Multi-Query helps here:')
print('  A query about capital requirements AND lending criteria')
print('  maps to a centroid between two document clusters.')
print('  Sub-questions target each cluster independently.')

## Cell 3: Core — Query Decomposer, Retrieval, Aggregation, Pipeline
**What this demonstrates**: Three functions implement the full pattern: `decompose_query` breaks the complex question into sub-queries, `retrieve_for_subquery` fetches chunks per sub-query, and `multi_query_rag` orchestrates decompose → retrieve × N → deduplicate → generate.
**Expected output**: Function definitions confirmed.

In [ ]:
def decompose_query(complex_query: str, n: int = N_SUB_QUERIES) -> list[str]:
    """Break a complex multi-part question into n focused sub-queries.

    Each sub-query targets a distinct aspect of the original question,
    mapping to a tighter region of the embedding space than the full query.

    Args:
        complex_query: The user's multi-faceted question.
        n:             Number of sub-queries to generate.

    Returns:
        List of n focused sub-queries.
    """
    prompt = '\n'.join([
        f'Break the following complex question into {n} focused sub-questions.',
        'Each sub-question must:',
        '  1. Target a distinct aspect of the original question',
        '  2. Be self-contained and answerable on its own',
        '  3. Use specific, concrete terminology (not vague generalisations)',
        'Return one sub-question per line. No numbering, no prefixes.',
        '',
        f'Question: {complex_query}',
    ])
    response = client.messages.create(
        model=DECOMPOSE_MODEL,
        max_tokens=300,
        messages=[{'role': 'user', 'content': prompt}],
    )
    lines = [
        line.strip()
        for line in response.content[0].text.strip().splitlines()
        if line.strip()
    ]
    return lines[:n]


def retrieve_for_subquery(subquery: str, k: int = K_PER_QUERY) -> list[Document]:
    """Retrieve top-k chunks for a single sub-query.

    Args:
        subquery: A focused sub-question targeting one aspect of the original query.
        k:        Number of documents to retrieve.

    Returns:
        Ranked list of top-k Document objects.
    """
    return vectorstore.similarity_search(subquery, k=k)


def multi_query_rag(
    query: str,
    n_sub_queries: int = N_SUB_QUERIES,
    k_per_query: int   = K_PER_QUERY,
    max_context: int   = MAX_CONTEXT,
) -> dict:
    """Full Multi-Query RAG pipeline: decompose → retrieve × N → union → generate.

    The original query is always included in the retrieval pool — sub-queries
    expand coverage but do not replace the original.

    Args:
        query:         The user's complex multi-part question.
        n_sub_queries: Number of sub-queries to decompose into.
        k_per_query:   Chunks to retrieve per sub-query.
        max_context:   Maximum unique chunks to pass to the generator.

    Returns:
        dict with keys: query, sub_queries, all_queries, per_query_results,
                        aggregated, answer, latency_ms.
    """
    t0 = time.perf_counter()

    # Step 1: decompose into sub-queries; original always included
    sub_queries = decompose_query(query, n=n_sub_queries)
    all_queries = [query] + sub_queries

    # Step 2: retrieve top-k chunks for each query independently
    per_query_results: list[list[Document]] = [
        retrieve_for_subquery(q, k=k_per_query)
        for q in all_queries
    ]

    # Step 3: union + deduplicate — no rank weighting, all unique chunks treated equally
    seen: set[str] = set()
    aggregated: list[Document] = []
    for docs in per_query_results:
        for doc in docs:
            key = doc.page_content[:80]   # stable content identifier
            if key not in seen:
                seen.add(key)
                aggregated.append(doc)

    # Step 4: cap context to avoid overwhelming the generator
    context_docs = aggregated[:max_context]

    # Step 5: build context with source annotations
    context = '\n\n---\n\n'.join(
        f'[source={doc.metadata.get("source", "?")}]\n{doc.page_content}'
        for doc in context_docs
    )

    # Step 6: generate answer from aggregated context
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=500,
        system=(
            'Answer using only the provided context. '
            'Address each part of the question. '
            'Cite the source document for each claim.'
        ),
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}],
    )

    return {
        'query'             : query,
        'sub_queries'       : sub_queries,
        'all_queries'       : all_queries,
        'per_query_results' : per_query_results,
        'aggregated'        : aggregated,
        'context_docs'      : context_docs,
        'answer'            : response.content[0].text.strip(),
        'latency_ms'        : (time.perf_counter() - t0) * 1000,
    }


print('Multi-Query RAG pipeline defined')
print('  decompose_query()        — N sub-queries via Haiku')
print('  retrieve_for_subquery()  — per-sub-query similarity search')
print('  multi_query_rag()        — decompose → retrieve×N → union → generate')

## Cell 4: Run — Capital Requirements and Lending Criteria Query
**What this demonstrates**: A query that deliberately spans two documents — Basel III capital rules and fintech lending policy. Single-query retrieval anchors to one document; sub-question decomposition retrieves targeted sections from both.
**Expected output**: Original query, 3 generated sub-queries, and the final answer citing both source documents.

In [ ]:
QUERY = (
    'What are the capital requirements and how do they interact '
    'with loan eligibility criteria?'
)

print(f'Query: {QUERY}')
print()

result = multi_query_rag(QUERY)

# Show decomposition
print('Decomposed into sub-queries:')
print(f'  [Original]  {result["query"]}')
for i, sq in enumerate(result['sub_queries'], 1):
    print(f'  [Sub-query {i}] {sq}')
print()

# Retrieval summary
total_retrieved = sum(len(docs) for docs in result['per_query_results'])
print(f'Retrieval: {len(result["all_queries"])} queries × {K_PER_QUERY} = {total_retrieved} total')
print(f'After dedup: {len(result["aggregated"])} unique chunks')
print(f'Passed to generator: {len(result["context_docs"])} chunks (cap: {MAX_CONTEXT})')
print(f'Latency: {result["latency_ms"]:.0f} ms')
print()

# Source distribution in aggregated context
src_counts = Counter(d.metadata.get('source', '?') for d in result['context_docs'])
print('Source distribution in context:')
for src, cnt in src_counts.items():
    print(f'  {src:<18}: {cnt} chunks')
print()

print('=' * 65)
print('ANSWER')
print('=' * 65)
print(result['answer'])

## Cell 5: Inspect — Sub-Queries, Per-Query Chunks, Aggregated Context
**What this demonstrates**: The mechanics of decomposition — which sub-query retrieved which chunks, how many unique chunks each sub-query contributed to the final context, and what a single-query baseline would have missed.
**Expected output**: Per-sub-query chunk previews, source breakdown, and baseline comparison.

In [ ]:
# ── Per-sub-query retrieval preview ───────────────────────────────────────────
print('PER-SUB-QUERY RETRIEVAL (top 3 each)')
print('=' * 70)
for label, docs in zip(result['all_queries'], result['per_query_results']):
    tag = 'Original' if label == QUERY else 'Sub-query'
    print(f'\n[{tag}] {label[:62]}')
    for rank, doc in enumerate(docs, 1):
        src     = doc.metadata.get('source', '?')
        preview = doc.page_content[:70].replace('\n', ' ')
        print(f'  {rank}. [{src}] {preview}...')

# ── Unique contribution per sub-query ─────────────────────────────────────────
print()
print('UNIQUE CHUNK CONTRIBUTION PER SUB-QUERY')
print('=' * 70)
# Build the union incrementally to show marginal contribution
running_seen: set[str] = set()
for label, docs in zip(result['all_queries'], result['per_query_results']):
    tag       = 'Original' if label == QUERY else 'Sub-query'
    keys      = {d.page_content[:80] for d in docs}
    new_keys  = keys - running_seen
    running_seen.update(keys)
    src_breakdown = Counter(d.metadata.get('source','?') for d in docs if d.page_content[:80] in new_keys)
    print(f'  [{tag:<9}] +{len(new_keys)} new chunks  {dict(src_breakdown)}')
print(f'  Total unique: {len(running_seen)} chunks')

# ── Aggregated context source breakdown ───────────────────────────────────────
print()
print('AGGREGATED CONTEXT — SOURCE BREAKDOWN')
print('=' * 70)
for src, cnt in Counter(d.metadata.get('source','?') for d in result['aggregated']).items():
    bar = '█' * cnt
    print(f'  {src:<18} {bar} ({cnt})')

# ── Baseline comparison ────────────────────────────────────────────────────────
print()
print('BASELINE: single-query retrieval vs Multi-Query aggregation')
print('=' * 70)
baseline_docs = vectorstore.similarity_search(QUERY, k=5)
baseline_keys = {d.page_content[:80] for d in baseline_docs}
agg_keys      = {d.page_content[:80] for d in result['aggregated']}
new_in_mq     = agg_keys - baseline_keys

print(f'  Baseline top-5       : {len(baseline_keys)} unique chunks')
print(f'  Multi-Query union    : {len(agg_keys)} unique chunks')
print(f'  New in Multi-Query   : {len(new_in_mq)} chunks baseline missed')
print()

baseline_srcs = Counter(d.metadata.get('source','?') for d in baseline_docs)
print('Baseline source coverage:', dict(baseline_srcs))
agg_srcs = Counter(d.metadata.get('source','?') for d in result['aggregated'])
print('Multi-Query coverage   :', dict(agg_srcs))

## Cell 6: Fintech — Multi-Framework Compliance Query
**What this demonstrates**: Multi-Query RAG on a compliance query that spans two regulatory frameworks (Basel III capital adequacy and AML/fintech policy). A single embedding cannot represent both frameworks simultaneously; sub-question decomposition retrieves precise sections from each, giving the compliance analyst a complete picture.
**Expected output**: Sub-queries targeting distinct regulatory frameworks, multi-source aggregated context, and a compliance answer that cites both frameworks.

In [ ]:
COMPLIANCE_QUERY = (
    'What are our regulatory capital obligations under Basel III '
    'and our transaction monitoring requirements under our fintech policy?'
)

print('FINTECH: MULTI-FRAMEWORK COMPLIANCE QUERY')
print(f'Query: {COMPLIANCE_QUERY}')
print()
print('Why this needs Multi-Query RAG:')
print('  Basel III → capital ratios, risk-weighted assets, buffer requirements')
print('  Fintech policy → transaction monitoring, suspicious activity, AML rules')
print('  A single embedding sits between two clusters and retrieves neither well.')
print()

compliance_result = multi_query_rag(COMPLIANCE_QUERY)

# Show sub-query decomposition with implied framework
print('Sub-query decomposition (each targets one framework):')
print(f'  [Original]  {compliance_result["query"][:65]}')
for i, sq in enumerate(compliance_result['sub_queries'], 1):
    print(f'  [Sub-query {i}] {sq}')
print()

# Source coverage per sub-query
print('Source coverage per sub-query:')
for label, docs in zip(compliance_result['all_queries'], compliance_result['per_query_results']):
    tag   = 'Original' if label == COMPLIANCE_QUERY else 'Sub-query'
    srcs  = sorted({d.metadata.get('source','?') for d in docs})
    print(f'  [{tag:<9}] {label[:50]}')
    print(f'             sources retrieved: {", ".join(srcs)}')
print()

# Aggregated context statistics
agg_src_counts = Counter(d.metadata.get('source','?') for d in compliance_result['aggregated'])
print(f'Aggregated context: {len(compliance_result["aggregated"])} unique chunks')
for src, cnt in agg_src_counts.items():
    print(f'  {src:<18}: {cnt} chunks')
print(f'Latency: {compliance_result["latency_ms"]:.0f} ms')
print()

print('=' * 65)
print('COMPLIANCE ANSWER')
print('=' * 65)
print(compliance_result['answer'])
print()
print('-' * 65)
print('Multi-Query RAG value for compliance:')
print('  Each sub-query retrieves a targeted cluster from one framework')
print('  Union gives the analyst complete cross-framework coverage')
print('  No manual synonym tables or query reformulation needed')